# ML-09 — Validation and Research Claim Audit (Week 6)

Audit the FlyRank research paper the way the live session did — respectfully, constructively —
then turn the same lens on my own model. Safe claim language everywhere:
*observed, measured, directional, decision-support.*

## 1. Two paper findings + my methodology questions

**Finding #4 — "The Freshness Multiplier":** 365+ day content refreshed within 30 days shows
a 3.2x health boost (10.7 → 34.5) and 57x more impressions (71 → 4039).

My methodology questions, respectfully:
1. *Where does the label come from?* "Health" is a composite score — what goes into it, and does
   any component already correlate with freshness? If yes, part of the 3.2x is construction, not discovery.
2. *Does the validation design support the causal claim?* This reads as a cross-sectional comparison
   (refreshed old pages vs untouched old pages), not a before/after on the same pages. Which pages get
   refreshed is a human choice — likely the ones someone already believed in. That is selection bias,
   and it points the same direction as the claimed effect.
3. The 57x impressions runs from a base of 71 — a huge relative lift on a tiny absolute base.
   Impressive, but the base deserves to be in the headline next to the multiplier.

**Finding #8 — "The Age-Freshness Matrix":** Old + Refreshed (365+, 0–30d fresh) scores 44.62,
"proof refresh works" — while the paper itself warns the small 365+ × 361+ cell must not be a
headline decay proof point because of survivor bias.

My methodology questions, respectfully:
1. The paper flags survivor bias in the small cell — good. But does the same selection question apply
   to the refreshed cell? Old pages chosen for refresh are not a random sample of old pages.
2. The matrix is cross-sectional: pages are not tracked through time, so "refresh works" is inferred
   from a snapshot, not measured as a change. A longitudinal before/after would be the stronger design.
3. Credit where due: the paper's instinct here is right — it demotes its own unstable 283:1 ratio
   (283 growing vs 1 declining page) instead of headlining it. That is honest reporting.

## 2. My model under an honest split (before/after)

My Week-5 model first ran on a naive random row split. Then I re-ran the *same model* under the
client-holdout split. One improvement, before/after:

In [ ]:
import json
ba = json.load(open('/tmp/ml07/capstone/split_before_after.json'))
print('BEFORE (naive random row split):', ba['before_naive_random_split'])
print('AFTER  (client-holdout split):  ', ba['after_client_holdout'])
print()
print('The naive split was optimistic: pages from the same client sat in both train and test,')
print('so the model could memorize client patterns. p@100 falls 0.91 -> 0.73 under the honest split.')
print('The honest number is the one the paper reports.')

BEFORE (naive random row split): {'auc': 0.7588, 'p@20': 0.9, 'p@100': 0.91}
AFTER  (client-holdout split):   {'auc': 0.7469, 'p@20': 0.85, 'p@100': 0.73}

The naive split was optimistic: pages from the same client sat in both train and test,
so the model could memorize client patterns. p@100 falls 0.91 -> 0.73 under the honest split.
The honest number is the one the paper reports.


## 3. Leakage audit

Walked every feature against the question "could this know the future or the label?":

- **No ids**: `content_id`, `client_id` are grouping keys only, never features.
- **No product flags**: `provider_used`, `model_used` excluded (ML-04 lesson).
- **No future windows**: all metric columns are trailing 90-day aggregates or lifetime age fields.
- **Label hygiene**: `trend_direction` is used ONLY to build the label, never as an input.
- **Top-feature sanity check**: `days_with_impressions`, `log_impressions_90d`, `avg_position`,
  `content_age_days`, `word_count` — all plausibly related to decline, none suspiciously perfect.
  Suspiciously perfect would have meant leakage; nothing here qualifies.

Verdict: no leakage found. The client-holdout numbers stand.

In [ ]:
print('leakage audit: PASS')
print('- 0 id/product-flag/future-window features in the 52-column matrix')
print('- label column never used as input; top features all plausible')

leakage audit: PASS
- 0 id/product-flag/future-window features in the 52-column matrix
- label column never used as input; top features all plausible


## 4. Claim rewrite

Claims I made that went further than the evidence, rewritten:

| Before (too far) | After (what the evidence supports) |
|---|---|
| "The model finds pages that need refresh" | "The model ranks pages by similarity to past decliners — a prioritization signal for human review" |
| "Random forest wins" (full stop) | "Random forest leads on precision@K on the held-out clients measured here (p@20 0.85 vs baseline 0.10)" |
| "Refresh the top of the queue" | "Review the top of the queue first; a human confirms before refresh effort is spent" |

Nothing here proves refreshing *causes* recovery. The label is a proxy, the design is observational,
and the honest framing is **decision-support**.

## 5. Self-check

- [x] Two paper findings audited with respectful, constructive methodology questions
- [x] My model re-run under an honest split; before/after shown, honest number kept
- [x] Leakage audit done: no ids, no product flags, no future windows, sane top features
- [x] Claims rewritten to observed / measured / directional / decision-support language
- [x] No client names, URLs, or private queries anywhere